# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MUKUL-TIWARI/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method choice

I selected a Random Forest classifier because it can learn non-linear relationships between search visibility, ranking position, and engagement without requiring complex feature engineering. It also provides feature importance, making the model easier to interpret and compare against the Week-4 baseline.

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn


In [6]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL="hf://datasets/FlyRank/internship-warehouse"

TABLES={
    "fact_daily":f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

query=f"""
SELECT
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_clicks,
gsc_avg_position,
COALESCE(ga4_engaged_sessions,0) AS ga4_engaged_sessions
FROM {TABLES["fact_daily"]}
WHERE month='2026-03'
AND gsc_impressions>=100
LIMIT 300000
"""

df=con.sql(query).df()
print(df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                 client_hash_id           content_hash_id  gsc_impressions  \
0       client_73cda7b4e4f265ea  content_7a105f548d9c6916              125   
1       client_73cda7b4e4f265ea  content_36c36abc7650d7af              239   
2       client_73cda7b4e4f265ea  content_a7da352b73b02668              191   
3       client_73cda7b4e4f265ea  content_712c365258cee05c              223   
4       client_73cda7b4e4f265ea  content_aafb2ab7e5fc80d0              126   
...                         ...                       ...              ...   
299995  client_20259bd6705d81d4  content_e3286b310daacf2e              201   
299996  client_20259bd6705d81d4  content_1b7a045a9a81c086              112   
299997  client_20259bd6705d81d4  content_35f08e455460af20              199   
299998  client_20259bd6705d81d4  content_fb7b5a47812a52ae              260   
299999  client_20259bd6705d81d4  content_ac0f6650a3eb74a9              141   

        gsc_clicks  gsc_avg_position  ga4_engaged_sessions  
0 

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split design

A random train/test split was used to evaluate the first modeling approach. The same dataset and target definition are used for both the baseline and the Random Forest model so that performance can be compared fairly.

In [3]:
df["label"]=(df["ga4_engaged_sessions"]==0).astype(int)

X=df[
[
"gsc_impressions",
"gsc_clicks",
"gsc_avg_position"
]
]

y=df["label"]

X_train,X_test,y_train,y_test=train_test_split(
X,
y,
test_size=0.2,
random_state=42,
stratify=y
)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Model vs baseline

The Random Forest model was compared against the transparent Week-4 baseline using the same data split and the same prediction target.

In [4]:
baseline_pred=(X_test["gsc_avg_position"]>20).astype(int)

rf=RandomForestClassifier(
n_estimators=200,
random_state=42,
n_jobs=-1
)

rf.fit(X_train,y_train)

model_pred=rf.predict(X_test)

results=pd.DataFrame({

"Model":[
"Week4 Baseline",
"Random Forest"
],

"Accuracy":[
accuracy_score(y_test,baseline_pred),
accuracy_score(y_test,model_pred)
],

"Precision":[
precision_score(y_test,baseline_pred),
precision_score(y_test,model_pred)
],

"Recall":[
recall_score(y_test,baseline_pred),
recall_score(y_test,model_pred)
],

"F1":[
f1_score(y_test,baseline_pred),
f1_score(y_test,model_pred)
]

})

results


,Model,Accuracy,Precision,Recall,F1
0,Week4 Baseline,0.221067,0.956643,0.210498,0.345067
1,Random Forest,0.971900,0.975650,0.996034,0.985736


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and interpretation

The Random Forest reduced errors compared with the baseline by combining multiple search signals instead of relying on a single rule. Feature importance suggests that search impressions, ranking position, and clicks contribute most to the predictions. Some errors are expected because engagement can also depend on content quality, seasonality, and user intent, which are not included in this baseline feature set.

In [5]:
importance=pd.DataFrame({

"Feature":X.columns,

"Importance":rf.feature_importances_

}).sort_values("Importance",ascending=False)

importance

,Feature,Importance
2,gsc_avg_position,0.630018
0,gsc_impressions,0.300864
1,gsc_clicks,0.069119


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.